In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('양치전처리.csv')
df.head(10)

# columns : 18 , row : 11898
# target : NO_CPI_34 [치주질환 여부]

,NO_CPI_34,age,sex,BS3_1,DE1_dg,BD1_11,OR1_2,DF2_pr,BP1,Total_slp_wk,Total_slp_wd,BM1,BM2_1,BM2_2,BM2_3,BM2_4,BM2_5,HE_BMI
0,0.0,80.0,2.0,4.0,0.0,4.0,0.0,0.0,4.0,600.0,600.0,2.0,0.0,0.0,0.0,0.0,0.0,2.0
1,0.0,59.0,2.0,4.0,0.0,2.0,1.0,0.0,3.0,420.0,360.0,4.0,1.0,1.0,1.0,0.0,0.0,2.0
2,0.0,68.0,1.0,1.0,0.0,6.0,0.0,0.0,3.0,510.0,510.0,3.0,0.0,0.0,1.0,0.0,0.0,2.0
3,0.0,64.0,2.0,4.0,1.0,0.0,1.0,0.0,3.0,240.0,240.0,2.0,1.0,1.0,0.0,0.0,0.0,3.0
4,0.0,77.0,1.0,3.0,1.0,2.0,0.0,0.0,4.0,480.0,480.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0
5,1.0,68.0,2.0,4.0,0.0,0.0,0.0,0.0,4.0,360.0,300.0,2.0,0.0,0.0,1.0,0.0,0.0,3.0
6,1.0,66.0,1.0,3.0,0.0,2.0,0.0,0.0,3.0,480.0,480.0,3.0,0.0,0.0,0.0,0.0,0.0,2.0
7,0.0,60.0,2.0,4.0,0.0,2.0,1.0,0.0,3.0,600.0,660.0,3.0,1.0,0.0,0.0,0.0,0.0,3.0
8,1.0,47.0,1.0,4.0,0.0,5.0,1.0,0.0,3.0,300.0,420.0,2.0,1.0,0.0,0.0,0.0,0.0,2.0
9,0.0,45.0,2.0,4.0,0.0,3.0,0.0,0.0,2.0,390.0,540.0,2.0,0.0,0.0,0.0,0.0,0.0,2.0


#### 주요 파라미터
- 커널(kernel) : 판별 경계를 찾기위한 내적 함수, 'linear', 'poly', 'rbf', 'sigmoid', 'precomputed', default는 rdf
- C : 잘못 분류된 데이터로부터 해당 클래스의 결정 경계까지의 거리로 데이터점이 잘못 분류된 정도를 나타냄
    - SVM 모델이 오류를 어느정도 허용할 것인지 파라미터 S를 통해 지정
    - C값이 클 수록 하드마진(오류 허용 안 함), 작을 수록 소프트마진(오류를 허용함)
- gamma : 결정 경계를 얼마나 유연하게 그을 것인지 정해주는 역할
    - gamma값을 높이면 학습 데이터에 많이 의존해서 결정 경계를 구불구불 긋게 된다. 오버피팅을 초래
    - gamma값을 낮추면 학습 데이터에 별로 의존하지 않고 결정 경계를 직선에 가깝게 긋게 된다. 언더피팅 초래

In [5]:
# 기본 모델

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# 1. 특성(X)과 타겟(y) 분리
# 타겟 컬럼인 'NO_CPI_34'를 제외한 나머지를 X로 설정
X = df.drop('NO_CPI_34', axis=1)
y = df['NO_CPI_34']

# 2. 학습 데이터와 테스트 데이터 분리 (8:2 비율)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. 데이터 스케일링 (SVM은 스케일링이 매우 중요합니다)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. SVM 모델 생성 및 학습
# 데이터 양이 어느 정도(약 1.2만건) 있으므로 기본 RBF 커널을 사용합니다.
model = SVC(kernel='rbf', random_state=42)
model.fit(X_train_scaled, y_train)

# 5. 예측 및 결과 출력
y_pred = model.predict(X_test_scaled)

print("### SVM 모델 평가 결과 ###")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\n[Detail Report]")
print(classification_report(y_test, y_pred))

### SVM 모델 평가 결과 ###
Accuracy: 0.7173

[Detail Report]
              precision    recall  f1-score   support

         0.0       0.74      0.92      0.82      1659
         1.0       0.60      0.27      0.37       746

    accuracy                           0.72      2405
   macro avg       0.67      0.59      0.59      2405
weighted avg       0.69      0.72      0.68      2405



In [ ]:
# 1. 그리드 cv 활용
# 2. class_weight + balanced 추가
# 3. c : 너무 크면 오버피팅, 너무 작으면 언더피팅이므로 [0.1, 1, 10] 의 범위 탐색
# 4. gamma : 결정 경계의 곡률 조정

In [8]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# 1. 특성(X)과 타겟(y) 분리
X = df.drop('NO_CPI_34', axis=1)
y = df['NO_CPI_34']

# 2. 학습/테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. 데이터 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. 하이퍼파라미터 튜닝 (GridSearchCV)
# C와 gamma의 범위를 지정하여 최적의 조합을 찾습니다.
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1],  # 1보다 작은 값 위주로 탐색
    'kernel': ['rbf']
}
# class_weight='balanced'를 추가하여 데이터 불균형 문제를 해결합니다. (Recall 향상 핵심)
grid = GridSearchCV(SVC(class_weight='balanced', random_state=42), param_grid, refit=True, verbose=2, cv=3)
grid.fit(X_train_scaled, y_train)

# 최적의 파라미터 출력
print(f"Best Parameters: {grid.best_params_}")

# 5. 예측 및 결과 출력
grid_predictions = grid.predict(X_test_scaled)

print("\n### 최적화된 SVM 모델 평가 결과 ###")
print(f"Accuracy: {accuracy_score(y_test, grid_predictions):.4f}")
print("\n[Detail Report]")
print(classification_report(y_test, grid_predictions))

Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END .....................C=0.1, gamma=0.001, kernel=rbf; total time=   3.1s
[CV] END .....................C=0.1, gamma=0.001, kernel=rbf; total time=   3.0s
[CV] END .....................C=0.1, gamma=0.001, kernel=rbf; total time=   2.6s
[CV] END ......................C=0.1, gamma=0.01, kernel=rbf; total time=   2.7s
[CV] END ......................C=0.1, gamma=0.01, kernel=rbf; total time=   3.0s
[CV] END ......................C=0.1, gamma=0.01, kernel=rbf; total time=   2.8s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=   2.7s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=   2.0s
[CV] END .......................C=0.1, gamma=0.1, kernel=rbf; total time=   2.5s
[CV] END .......................C=1, gamma=0.001, kernel=rbf; total time=   2.6s
[CV] END .......................C=1, gamma=0.001, kernel=rbf; total time=   2.7s
[CV] END .......................C=1, gamma=0.001